In [1]:
!pip install transformers datasets peft accelerate bitsandbytes trl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 47.5 MB/s eta 0:00:00


In [2]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from datasets import load_dataset
from torch.utils.data import DataLoader
import torch

In [3]:
# Loading GPT-2 in 4-bit

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type ='nf4',
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_use_double_quant = True
)

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    "gpt2",
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded.


In [4]:
# Attach LoRA
model = prepare_model_for_kbit_training(model) #

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["c_attn", "c_proj"],  # GPT-2 specific names
)

model = get_peft_model(model, lora_config) # inject A and B matrices in the GPT - 2 model
model.print_trainable_parameters()

trainable params: 1,622,016 || all params: 126,061,824 || trainable%: 1.2867


In [25]:
# Load and format IMDB dataset
dataset = load_dataset("imdb", split="train")
# Use 2000 examples — enough to see clear behavior change on T4
# without exceeding the free tier time limit (~30 min training)

dataset = dataset.shuffle(seed=42).select(range(2000))

def format_example(example):
  label = "POSITIVE" if example["label"]==1 else "NEGATIVE"
  text  = f"<|sentiment|>{label}<|review|>{example['text']}{tokenizer.eos_token}"
  return {"text": text}

dataset = dataset.map(format_example, remove_columns=["text", "label"])


In [20]:
print("Sample formatted example:")
print(dataset[1]["text"])

Sample formatted example:
<|sentiment|>POSITIVE<|review|>This movie is a great. The plot is very true to the book which is a classic written by Mark Twain. The movie starts of with a scene where Hank sings a song with a bunch of kids called "when you stub your toe on the moon" It reminds me of Sinatra's song High Hopes, it is fun and inspirational. The Music is great throughout and my favorite song is sung by the King, Hank (bing Crosby) and Sir "Saggy" Sagamore. OVerall a great family movie or even a great Date movie. This is a movie you can watch over and over again. The princess played by Rhonda Fleming is gorgeous. I love this movie!! If you liked Danny Kaye in the Court Jester then you will definitely like this movie.<|endoftext|>


In [26]:
# Tokenize
def tokenize(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )
    result["labels"] = [
        t if t != tokenizer.pad_token_id else -100
        for t in result["input_ids"]
    ]
    return result

tokenized = dataset.map(tokenize, remove_columns=["text"])
tokenized.set_format("torch")
print(f"Dataset size: {len(tokenized)} examples")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset size: 2000 examples


In [29]:
#training loop
train_loader = DataLoader(tokenized, batch_size = 40, shuffle = True)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-4,
    weight_decay = 0.01
)

num_epochs = 10
total_steps = len(train_loader) * num_epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps = 20,
    num_training_steps = total_steps
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

model.train()

for epoch in range(num_epochs):
    total_loss = 0

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids = input_ids,
            attention_mask = attention_mask,
            labels = labels
        )

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()

        if step % 50 == 0:
          print(f'Epoch {epoch+1}/{num_epochs}, Step {step}/{len(train_loader)}, Loss: {loss.item():.4f}')

    avg = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} complete — avg loss: {avg:.4f}\n")

print("Training done.")


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Training on: cuda


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
/tmp/ipykernel_1228/1506341173.py:42: FutureWarning: `torch.nn.utils.clip_grad_norm` is now deprecated in favor of `torch.nn.utils.clip_grad_norm_`.
  torch.nn.utils.clip_grad_norm(model.parameters(), 1.0)


Epoch 1/10, Step 0/50, Loss: 4.2627

Epoch 1 complete — avg loss: 3.9489

Epoch 2/10, Step 0/50, Loss: 3.7486

Epoch 2 complete — avg loss: 3.6025

Epoch 3/10, Step 0/50, Loss: 3.6234

Epoch 3 complete — avg loss: 3.5512

Epoch 4/10, Step 0/50, Loss: 3.5093

Epoch 4 complete — avg loss: 3.5315

Epoch 5/10, Step 0/50, Loss: 3.4632

Epoch 5 complete — avg loss: 3.5182

Epoch 6/10, Step 0/50, Loss: 3.4984

Epoch 6 complete — avg loss: 3.5151

Epoch 7/10, Step 0/50, Loss: 3.3962

Epoch 7 complete — avg loss: 3.5075

Epoch 8/10, Step 0/50, Loss: 3.3612

Epoch 8 complete — avg loss: 3.5035

Epoch 9/10, Step 0/50, Loss: 3.4166

Epoch 9 complete — avg loss: 3.5005

Epoch 10/10, Step 0/50, Loss: 3.4261

Epoch 10 complete — avg loss: 3.4981

Training done.


In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
# import google drive and store the adapter
# ── Cell 8: Save adapter ──────────────────────────────────────────────
model.save_pretrained("/content/drive/MyDrive/Research Papers/LoRA_imdb")
tokenizer.save_pretrained("/content/drive/MyDrive/Research Papers/LoRA_imdb")
print("Adapter saved.")

Adapter saved.


In [36]:
# Inference
model.eval()

def generate_review(sentiment="POSITIVE", max_new_tokens=150):
    prompt = f"<|sentiment|>{sentiment}<|review|>"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            top_k=50,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,  # discourages repeating phrases
        )

    full_output = tokenizer.decode(output_ids[0], skip_special_tokens=False)
    # Extract only the review part after the <|review|> tag
    review = full_output.split("<|review|>")[-1].replace(tokenizer.eos_token, "").strip()
    return review

# Compare base model vs fine-tuned
print("=" * 50)
print("POSITIVE review:")
print(generate_review("POSITIVE"))

print("\n" + "=" * 50)
print("NEGATIVE review:")
print(generate_review("NEGATIVE"))

POSITIVE review:
I've seen this movie many times, and the first one I thought was probably not worth a second. It's funny how it takes place in an alternate reality where there are no human beings on earth to help humans or something.<br /><strong>"A man who works at McDonalds can't understand why his daughter is having trouble with her boyfriend when she has been dating him for six months."<b />This film feels like its taken from any other video game (like Star Trek) except that you're still allowed play through some of those boring "stories" as they come along... but then again your character doesn`t even know what he/she did wrong! The same thing happened over and Over here . You have lots more options

NEGATIVE review:
If you want a movie about the story of "The Great Escape", then this is it. And if not, why did people stop watching? It's very bad! I was hoping to find out more information on that film and how terrible its performance really was.<br /><div id="selection" style="" 

In [38]:
# Merging
# Only needed if you want zero-latency inference.

base = AutoModelForCausalLM.from_pretrained(
    "gpt2",
    torch_dtype=torch.float16,
    device_map="auto",
)

merged = PeftModel.from_pretrained(base, "/content/drive/MyDrive/Research Papers/LoRA_imdb")
merged = merged.merge_and_unload()

merged.save_pretrained("/content/drive/MyDrive/Research Papers/LoRA_LoRA_imdb_merged")
print("Merged model saved.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved.


In [42]:
# ── Comparison Cell ───────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Load base GPT-2 ─────────────────────────────────
base_model = AutoModelForCausalLM.from_pretrained(
    "gpt2",
    torch_dtype=torch.float16,
    device_map="auto",
)
base_model.eval()

# ── Load fine-tuned model ────────────────
finetuned_model = AutoModelForCausalLM.from_pretrained(
    "/content/drive/MyDrive/Research Papers/LoRA_imdb/LoRA_imdb_merged",
    torch_dtype=torch.float16,
    device_map="auto",
)
finetuned_model.eval()

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [44]:
# ── Side-by-side comparison ───────────────────────────────────────────

prompts = [
    {
        "label"     : "Positive review prompt",
        "base"      : "<|sentiment|>POSITIVE<|review|>",
        "finetuned" : "<|sentiment|>POSITIVE<|review|>",
    },
    {
        "label"     : "Negative review prompt",
        "base"      : "<|sentiment|>NEGATIVE<|review|>",
        "finetuned" : "<|sentiment|>NEGATIVE<|review|>",
    },
    {
        "label"     : "Neutral text prompt (no special tokens)",
        "base"      : "This movie was",
        "finetuned" : "This movie was",
    },
]

for p in prompts:
    base_out      = generate(base_model,      p["base"])
    finetuned_out = generate(finetuned_model, p["finetuned"])

    print("=" * 60)
    print(f"PROMPT TYPE : {p['label']}")
    print(f"PROMPT      : {p['base']!r}")
    print()
    print("── BASE MODEL ──────────────────────────────────────")
    print(base_out)
    print()
    print("── FINE-TUNED MODEL ────────────────────────────────")
    print(finetuned_out)
    print()

PROMPT TYPE : Positive review prompt
PROMPT      : '<|sentiment|>POSITIVE<|review|>'

── BASE MODEL ──────────────────────────────────────
TRANSFERANT</a></p><br/> <div class="c1" style="margin:0 0 100%;padding-bottom:-2px;" > </span></li>.
\r

. .listListBox { position:relative!important; fontsize=12em !important; margin :10%; } /* {{{fontfamily}}, CSS and JS styles are the default for all browsers */ div width = "15"><!-- Margin of text --> <!-- HTML code to be used in Listbox here (the size is 8 bytes) with 4 comments per

── FINE-TUNED MODEL ────────────────────────────────
It is the first film I have seen which has had a long, successful run in Australia. The story of its protagonist is very complex and not only consists mainly around three characters who are born from each other (the main character being Mr Bean). When they meet up with their parents after school for lunch at home, there's an awkward moment when we realise that it was one day before his birthday - he hasn't eaten

In [45]:
# ── Metric 1: Perplexity on held-out IMDB reviews ────────────────────
# Lower perplexity = model finds these reviews more "natural"
# Fine-tuned model should have lower perplexity on IMDB text

import math

def compute_perplexity(model, text, max_length=256):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    ).to(device)

    labels = inputs["input_ids"].clone()

    with torch.no_grad():
        outputs = model(**inputs, labels=labels)

    # loss is mean negative log likelihood per token
    return math.exp(outputs.loss.item())

# Test on a real IMDB review not seen during training
test_review = """This film is an absolute masterpiece. The performances
are haunting, the cinematography breathtaking, and the story left me
thinking for days. A rare gem that deserves every award it receives."""

base_ppl      = compute_perplexity(base_model,      test_review)
finetuned_ppl = compute_perplexity(finetuned_model, test_review)

print(f"Base model perplexity      : {base_ppl:.2f}")
print(f"Fine-tuned model perplexity: {finetuned_ppl:.2f}")
print(f"Improvement                : {((base_ppl - finetuned_ppl) / base_ppl * 100):.1f}%")

Base model perplexity      : 54.28
Fine-tuned model perplexity: 61.35
Improvement                : -13.0%


In [46]:
# ── Metric 2: Sentiment consistency check ─────────────────────────────
# Generate N reviews per sentiment and count how many
# actually sound positive/negative using keyword matching
# (crude but fast — no extra model needed)

positive_keywords = ["great", "excellent", "amazing", "loved", "brilliant",
                     "wonderful", "fantastic", "perfect", "best", "outstanding"]
negative_keywords = ["terrible", "awful", "boring", "worst", "disappointing",
                     "horrible", "bad", "poor", "waste", "dreadful"]

def sentiment_score(text):
    text  = text.lower()
    pos   = sum(1 for w in positive_keywords if w in text)
    neg   = sum(1 for w in negative_keywords if w in text)
    if pos > neg:   return "POSITIVE"
    if neg > pos:   return "NEGATIVE"
    return "NEUTRAL"

n_samples = 10
results   = {"base": {"correct": 0}, "finetuned": {"correct": 0}}

for sentiment in ["POSITIVE", "NEGATIVE"]:
    prompt = f"<|sentiment|>{sentiment}<|review|>"
    for _ in range(n_samples):
        base_out      = generate(base_model,      prompt, max_new_tokens=80)
        finetuned_out = generate(finetuned_model, prompt, max_new_tokens=80)

        if sentiment_score(base_out)      == sentiment:
            results["base"]["correct"]      += 1
        if sentiment_score(finetuned_out) == sentiment:
            results["finetuned"]["correct"] += 1

total = n_samples * 2
print(f"Base model sentiment accuracy      : {results['base']['correct']}/{total}")
print(f"Fine-tuned model sentiment accuracy: {results['finetuned']['correct']}/{total}")

Base model sentiment accuracy      : 2/20
Fine-tuned model sentiment accuracy: 3/20


In [47]:
# ── Metric 3: Output diversity ────────────────────────────────────────
# Generate the same prompt 5 times and measure how different
# the outputs are from each other — fine-tuned models should
# be more focused but not repetitive

def average_unique_words(model, prompt, n=5):
    outputs     = [generate(model, prompt, max_new_tokens=60) for _ in range(n)]
    all_words   = [set(o.lower().split()) for o in outputs]
    # average number of unique words across all outputs combined
    total_unique = len(set.union(*all_words))
    avg_per_gen  = sum(len(w) for w in all_words) / n
    return total_unique, avg_per_gen, outputs

prompt = "<|sentiment|>POSITIVE<|review|>"

base_unique, base_avg, base_outs = average_unique_words(base_model, prompt)
ft_unique,   ft_avg,   ft_outs   = average_unique_words(finetuned_model, prompt)

print("BASE MODEL — 5 generations of same prompt:")
for i, o in enumerate(base_outs):
    print(f"  [{i+1}] {o[:80]}...")

print("\nFINE-TUNED — 5 generations of same prompt:")
for i, o in enumerate(ft_outs):
    print(f"  [{i+1}] {o[:80]}...")

print(f"\nBase model     — unique vocab across 5 runs: {base_unique}, avg words/gen: {base_avg:.0f}")
print(f"Fine-tuned     — unique vocab across 5 runs: {ft_unique},  avg words/gen: {ft_avg:.0f}")

BASE MODEL — 5 generations of same prompt:
  [1] LINKS <|url|href|>SHOWING_TIME</a> |
. . >RATING, +(0-5) (3 reviews), -1 comment...
  [2] DATE: February 13th
Categories : Science Fiction, Fantasy, Horror, Romance, Thri...
  [3] SECTION</subreddit_id>.
- In addition to the standard "C" and "A", each of these...
  [4] INFORMATIVE</textarea></p><table width="100%" cellpadding='0" border-top='' padd...
  [5] TRANSCRIPT[0]]
A [](HTTP_INVALID) HTTP/1.01 200 OK Content-Length: 749 bytes (5,...

FINE-TUNED — 5 generations of same prompt:
  [1] I am not a huge fan of the film, but I do like it. The cast and crew are all gre...
  [2] I saw this movie a couple of times at the cinema but it was mostly just one man....
  [3] I would like to say that I can't really tell you what the rating is of this movi...
  [4] This is a decent film, and I'm glad it's called this. But if you're into comedie...
  [5] I'm a huge fan of this movie. It's the only one I've ever seen that is so good! ...

Base model  